# 03 — Exploratory Data Analysis

**Dataset:** `data/processed/airbnb_nyc_cleaned.csv` — 48,884 listings, 27 columns  
**Goal:** Uncover trends, distributions, geographic patterns, host behaviour and seasonal signals that directly inform KPI design and the Tableau dashboard.

**Every chart cell is followed by a written business interpretation.**

| Section | Focus |
|---|---|
| 3.1 | Setup & data load |
| 3.2 | Univariate distributions — price, nights, reviews, availability |
| 3.3 | Borough-level comparisons |
| 3.4 | Room-type breakdown |
| 3.5 | Price-tier segmentation |
| 3.6 | Host behaviour — single vs multi-listers |
| 3.7 | Geographic density |
| 3.8 | Review activity & seasonality |
| 3.9 | Revenue proxy & occupancy analysis |
| 3.10 | Correlation heatmap |
| 3.11 | Outlier deep-dive |
| 3.12 | EDA Summary & hypotheses for notebook 04 |

---
## 3.1 Setup & Data Load

In [ ]:
from matplotlib.patches import Patch
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi'      : 120,
    'axes.titlesize'  : 13,
    'axes.titleweight': 'bold',
    'axes.labelsize'  : 11,
    'figure.titlesize': 15,
})

BOROUGH_PALETTE = {
    'Manhattan'    : '#E63946',
    'Brooklyn'     : '#457B9D',
    'Queens'       : '#2A9D8F',
    'Bronx'        : '#F4A261',
    'Staten Island': '#8338EC',
}
ROOM_PALETTE = {
    'Entire home/apt': '#1D3557',
    'Private room'   : '#457B9D',
    'Shared room'    : '#A8DADC',
}

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'airbnb_nyc_cleaned.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
df['last_review']  = pd.to_datetime(df['last_review'], errors='coerce')
df['price_tier']   = pd.Categorical(df['price_tier'],
                                     categories=['Budget','Mid-Range','Premium','Luxury'],
                                     ordered=True)


---
## 3.2 Univariate Distributions

### 3.2.1 Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['price'], bins=80, kde=True, color='#457B9D', ax=axes[0])
axes[0].set_title('Price Distribution (Raw)')
axes[0].set_xlabel('Price per Night ($)')
axes[0].axvline(df['price'].median(), color='#E63946', lw=2, ls='--',
                label=f'Median ${df["price"].median():.0f}')
axes[0].axvline(df['price'].mean(), color='#F4A261', lw=2, ls='--',
                label=f'Mean ${df["price"].mean():.0f}')
axes[0].legend()

sns.histplot(df['log_price'], bins=60, kde=True, color='#2A9D8F', ax=axes[1])
axes[1].set_title('Log(Price+1) Distribution — Normalised')
axes[1].set_xlabel('log1p(Price)')

fig.suptitle('Price Distribution: Raw vs Log-Transformed')
plt.tight_layout()
plt.show()


**Insight:** Price is heavily right-skewed (skewness ≈ 1.8). The median ($106) is significantly lower than the mean (~$153), confirming a long tail of premium listings pulling the average up. The log-transformed distribution is near-normal, which is why `log_price` will be used in regression and correlation analysis in notebook 04.

### 3.2.2 Minimum Nights Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

short = df[df['minimum_nights'] <= 30]
sns.histplot(short['minimum_nights'], bins=30, color='#1D3557', ax=axes[0])
axes[0].set_title('Minimum Nights (≤30 nights shown)')
axes[0].set_xlabel('Minimum Nights')

sorted_nights = np.sort(df['minimum_nights'].values)
cdf = np.arange(1, len(sorted_nights)+1) / len(sorted_nights)
axes[1].plot(sorted_nights, cdf, color='#E63946', lw=2)
axes[1].set_xlim(0, 30)
axes[1].set_title('CDF of Minimum Nights')
axes[1].set_xlabel('Minimum Nights')
axes[1].set_ylabel('Cumulative Proportion')
axes[1].axvline(7, color='#F4A261', ls='--', label='7 nights')
axes[1].axvline(30, color='#2A9D8F', ls='--', label='30 nights')
axes[1].legend()

plt.tight_layout()
plt.show()

pct_1   = (df['minimum_nights'] == 1).mean() * 100
pct_le7 = (df['minimum_nights'] <= 7).mean() * 100
pct_gt30 = (df['minimum_nights'] > 30).mean() * 100


**Insight:** The vast majority of listings (~85%) accept stays of ≤7 nights, confirming typical short-term rental behaviour. Listings requiring >30 nights (~8%) represent medium-term or quasi-residential rentals — a distinct segment that commands different pricing dynamics.

### 3.2.3 Review Count & Review Rate Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

active = df[df['has_reviews'] == 1]
sns.histplot(np.log1p(active['review_count']), bins=50,
             color='#457B9D', kde=True, ax=axes[0])
axes[0].set_title('log1p(Review Count) — Active Listings Only')
axes[0].set_xlabel('log1p(Review Count)')

sns.histplot(active['review_rate_month'].clip(upper=5), bins=50,
             color='#2A9D8F', kde=True, ax=axes[1])
axes[1].set_title('Reviews per Month (clipped at 5)')
axes[1].set_xlabel('Reviews per Month')

plt.tight_layout()
plt.show()


**Insight:** ~20.6% of listings have zero reviews, indicating either new listings or dormant/unbooked properties. Among active listings, the median review rate is ~1.4/month — meaning the average active listing receives roughly one booking per month at minimum.

### 3.2.4 Availability Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

sns.histplot(df['availability_365'], bins=60, color='#8338EC', kde=False, ax=ax)
ax.set_title('Availability Distribution (days/year)')
ax.set_xlabel('Available Days per Year')
ax.axvline(df['availability_365'].median(), color='#E63946', lw=2, ls='--',
           label=f'Median: {df["availability_365"].median():.0f} days')
ax.legend()
plt.tight_layout()
plt.show()


**Insight:** The bimodal distribution is striking — 35.9% of listings show 0 available days (fully blocked or removed from market) while 2.6% show 365 days. This bimodality separates *actively marketed* listings from dormant ones and is a critical filter for any dashboard showing real market supply.

---
## 3.3 Borough-Level Comparisons

In [ ]:
borough_order = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

counts = df['neighbourhood_group'].value_counts().reindex(borough_order)
bars = axes[0, 0].bar(counts.index, counts.values,
                       color=[BOROUGH_PALETTE[b] for b in counts.index])
axes[0, 0].set_title('Total Listings by Borough')
axes[0, 0].set_ylabel('Number of Listings')
for bar, val in zip(bars, counts.values):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                    f'{val:,}', ha='center', fontsize=9)

med_price = df.groupby('neighbourhood_group')['price'].median().reindex(borough_order)
bars2 = axes[0, 1].bar(med_price.index, med_price.values,
                        color=[BOROUGH_PALETTE[b] for b in med_price.index])
axes[0, 1].set_title('Median Nightly Price by Borough')
axes[0, 1].set_ylabel('Median Price ($)')
for bar, val in zip(bars2, med_price.values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'${val:.0f}', ha='center', fontsize=9)

box_data = [df[df['neighbourhood_group'] == b]['price'].clip(upper=500).values
            for b in borough_order]
bp = axes[1, 0].boxplot(box_data, labels=borough_order, patch_artist=True, notch=True)
for patch, b in zip(bp['boxes'], borough_order):
    patch.set_facecolor(BOROUGH_PALETTE[b])
    patch.set_alpha(0.7)
axes[1, 0].set_title('Price Distribution by Borough (clipped $500)')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].tick_params(axis='x', rotation=15)

occ = df.groupby('neighbourhood_group')['occupancy_rate_est'].mean().reindex(borough_order)
bars3 = axes[1, 1].bar(occ.index, occ.values * 100,
                        color=[BOROUGH_PALETTE[b] for b in occ.index])
axes[1, 1].set_title('Mean Estimated Occupancy Rate by Borough')
axes[1, 1].set_ylabel('Occupancy Rate (%)')
axes[1, 1].set_ylim(0, 100)
for bar, val in zip(bars3, occ.values):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, val*100 + 1,
                    f'{val*100:.1f}%', ha='center', fontsize=9)

fig.suptitle('Borough-Level Market Overview', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights:**
1. **Manhattan dominates supply and price** — 44.3% of all listings, median price $150 vs NYC median $106.
2. **Brooklyn is the volume challenger** — 41.1% of listings at a 35% discount to Manhattan, suggesting strong price-sensitive demand.
3. **Brooklyn has the highest occupancy (72.5%)**, matching Manhattan (69.3%) despite lower prices — this implies higher throughput and demand elasticity in Brooklyn.
4. **Staten Island has lowest occupancy (45.3%)** — geographic isolation drives lower utilisation despite moderate prices.

In [ ]:
rev_by_borough = (
    df.groupby('neighbourhood_group')
    .agg(
        listings       = ('id'             , 'count'),
        median_price   = ('price'          , 'median'),
        mean_price     = ('price'          , 'mean'),
        mean_occ       = ('occupancy_rate_est', 'mean'),
        mean_rev_proxy = ('revenue_proxy'  , 'mean'),
        total_rev_proxy= ('revenue_proxy'  , 'sum'),
        pct_multi      = ('is_multi_lister', 'mean'),
    )
    .reindex(borough_order)
    .round(2)
)
rev_by_borough['mean_occ'] = (rev_by_borough['mean_occ'] * 100).round(1).astype(str) + '%'
rev_by_borough['pct_multi'] = (rev_by_borough['pct_multi'] * 100).round(1).astype(str) + '%'
rev_by_borough

---
## 3.4 Room-Type Breakdown

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

room_order = ['Entire home/apt', 'Private room', 'Shared room']

room_counts = df['room_type'].value_counts().reindex(room_order)
axes[0].pie(
    room_counts.values,
    labels=room_counts.index,
    autopct='%1.1f%%',
    colors=[ROOM_PALETTE[r] for r in room_order],
    startangle=140,
    explode=(0.03, 0.03, 0.08),
)
axes[0].set_title('Listing Share by Room Type')

pivot = df.groupby(['neighbourhood_group', 'room_type'])['price'].median().unstack('room_type')
pivot = pivot.reindex(borough_order)[room_order]
pivot.plot(kind='bar', ax=axes[1], color=[ROOM_PALETTE[r] for r in room_order], rot=30, width=0.7)
axes[1].set_title('Median Price: Borough × Room Type')
axes[1].set_ylabel('Median Price ($)')
axes[1].legend(title='Room Type', bbox_to_anchor=(1.01, 1), loc='upper left')

occ_room = df.groupby('room_type')['occupancy_rate_est'].mean().reindex(room_order)
bars = axes[2].bar(occ_room.index,
                   occ_room.values * 100,
                   color=[ROOM_PALETTE[r] for r in room_order])
axes[2].set_title('Mean Occupancy Rate by Room Type')
axes[2].set_ylabel('Occupancy Rate (%)')
axes[2].set_ylim(0, 100)
for bar, val in zip(bars, occ_room.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, val*100 + 1,
                 f'{val*100:.1f}%', ha='center', fontsize=10)
axes[2].tick_params(axis='x', rotation=20)

fig.suptitle('Room-Type Market Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights:**
1. **Entire home/apt listings (52%)** generate 2–3× the revenue per booking but have lower occupancy than private rooms.
2. **Manhattan entire-home median = $191** — nearly double Brooklyn's $145, reflecting premium location demand.
3. **Shared rooms have surprisingly high occupancy** (~68%) despite lowest prices — driven by budget traveller demand in urban centres.

---
## 3.5 Price-Tier Segmentation

In [ ]:
tier_order  = ['Budget', 'Mid-Range', 'Premium', 'Luxury']
tier_colors = ['#A8DADC', '#457B9D', '#1D3557', '#E63946']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

tier_counts = df['price_tier'].value_counts().reindex(tier_order)
axes[0].bar(tier_counts.index, tier_counts.values, color=tier_colors)
axes[0].set_title('Listings per Price Tier')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(tier_counts.items()):
    axes[0].text(i, val + 50, f'{val:,}', ha='center', fontsize=9)

tier_by_borough = (
    df.groupby(['neighbourhood_group', 'price_tier'], observed=True)
    .size().unstack('price_tier')
    .reindex(borough_order)[tier_order]
)
tier_pct = tier_by_borough.div(tier_by_borough.sum(axis=1), axis=0) * 100
tier_pct.plot(kind='bar', stacked=True, ax=axes[1],
              color=tier_colors, rot=30, width=0.7)
axes[1].set_title('Price Tier Composition by Borough')
axes[1].set_ylabel('Share (%)')
axes[1].legend(title='Tier', bbox_to_anchor=(1.01, 1), loc='upper left')

occ_tier = df.groupby('price_tier', observed=True)['occupancy_rate_est'].mean().reindex(tier_order)
rev_tier  = df.groupby('price_tier', observed=True)['revenue_proxy'].mean().reindex(tier_order)
ax2 = axes[2].twinx()
axes[2].bar(tier_order, occ_tier.values * 100, color=tier_colors, alpha=0.7, label='Occupancy %')
ax2.plot(tier_order, rev_tier.values, color='#F4A261', lw=2.5, marker='o', ms=8, label='Avg Revenue Proxy')
axes[2].set_title('Occupancy & Revenue by Price Tier')
axes[2].set_ylabel('Occupancy Rate (%)')
ax2.set_ylabel('Avg Revenue Proxy ($)')
axes[2].set_ylim(0, 100)
lines1, labels1 = axes[2].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[2].legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

fig.suptitle('Price-Tier Segmentation Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights:**
1. **Each tier has ~12,000–12,400 listings** — near-equal quartile partitioning confirms clean segmentation.
2. **Manhattan has the highest Luxury/Premium share** — ~40% of Manhattan listings fall in the top two tiers.
3. **Budget listings dominate Bronx and Staten Island** — aligning with lower income demographics and lower tourist demand.
4. **Revenue proxy rises monotonically with tier**, but the occupancy gap between Budget and Luxury is small (~5%) — implying Luxury earns more primarily via higher price, not higher utilisation.

---
## 3.6 Host Behaviour — Single vs Multi-Listers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

host_labels = ['Single-Listing', 'Multi-Listing']
host_colors = ['#457B9D', '#E63946']

for val, label, color in zip([0, 1], host_labels, host_colors):
    subset = df[df['is_multi_lister'] == val]['price'].clip(upper=400)
    sns.kdeplot(subset, ax=axes[0], label=label, color=color, fill=True, alpha=0.3)
axes[0].set_title('Price Distribution: Single vs Multi-Lister')
axes[0].set_xlabel('Price ($)')
axes[0].legend()

multi_pct = (
    df.groupby('neighbourhood_group')['is_multi_lister'].mean() * 100
).reindex(borough_order).round(1)
axes[1].barh(multi_pct.index, multi_pct.values,
             color=[BOROUGH_PALETTE[b] for b in multi_pct.index])
axes[1].set_title('% Multi-Listing Hosts per Borough')
axes[1].set_xlabel('% of Listings from Multi-Listers')
axes[1].axvline(df['is_multi_lister'].mean()*100, ls='--', color='grey',
                label=f'Overall avg {df["is_multi_lister"].mean()*100:.1f}%')
axes[1].legend()

host_counts = df.groupby('host_id')['id'].count()
axes[2].hist(np.log1p(host_counts.values), bins=40, color='#8338EC', edgecolor='white')
axes[2].set_title('Distribution of log1p(Listings per Host)')
axes[2].set_xlabel('log1p(Listing Count)')
axes[2].set_ylabel('Number of Hosts')

fig.suptitle('Host Behaviour & Commercialisation Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


**Insights:**
1. **33.9% of listings are from multi-listing hosts** — a significant commercial/professional segment that likely manages properties as a business.
2. **Multi-listers price slightly higher** on average, possibly due to better optimisation or premium locations.
3. **Manhattan has the highest multi-lister concentration** — consistent with professional property management in high-value real estate markets.
4. **One host manages 327 listings** — a clear outlier representing hotel-like operations or property management firms.

---
## 3.7 Geographic Density — Top Neighbourhoods

In [ ]:
top_n = 20
top_hoods = df['neighbourhood'].value_counts().head(top_n)

hood_stats = (
    df[df['neighbourhood'].isin(top_hoods.index)]
    .groupby('neighbourhood')
    .agg(
        count        = ('id'             , 'count'),
        median_price = ('price'          , 'median'),
        mean_occ     = ('occupancy_rate_est', 'mean'),
        borough      = ('neighbourhood_group', 'first'),
    )
    .sort_values('count', ascending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

bar_colors = [BOROUGH_PALETTE[b] for b in hood_stats['borough']]
axes[0].barh(hood_stats.index, hood_stats['count'], color=bar_colors)
axes[0].set_title(f'Top {top_n} Neighbourhoods by Listing Count')
axes[0].set_xlabel('Number of Listings')

scatter_data = hood_stats.reset_index()
for _, row in scatter_data.iterrows():
    axes[1].scatter(row['median_price'], row['mean_occ'] * 100,
                    color=BOROUGH_PALETTE[row['borough']],
                    s=row['count'] / 15, alpha=0.8, edgecolors='white', lw=0.5)
    axes[1].annotate(row['neighbourhood'], (row['median_price'], row['mean_occ'] * 100),
                     fontsize=7, ha='left', va='bottom')
axes[1].set_xlabel('Median Price ($)')
axes[1].set_ylabel('Mean Occupancy Rate (%)')
axes[1].set_title('Price vs Occupancy by Neighbourhood\n(bubble = listing count)')

legend_elements = [Patch(facecolor=BOROUGH_PALETTE[b], label=b) for b in borough_order]
axes[1].legend(handles=legend_elements, title='Borough', fontsize=8)

plt.tight_layout()
plt.show()

**Insights:**
1. **Williamsburg (3,919) and Bedford-Stuyvesant (3,710)** are the two highest-supply neighbourhoods — both in Brooklyn, confirming Brooklyn's rise as a key Airbnb market.
2. **High price × high occupancy quadrant** (top-right of scatter) represents the most commercially successful micro-markets — typically Midtown and Hell's Kitchen.
3. **High price × lower occupancy** neighbourhoods (e.g. financial district) may be priced above the optimal demand curve.

---
## 3.8 Review Activity & Seasonality

In [ ]:
reviewed = df[df['last_review'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

year_counts = reviewed['review_year'].value_counts().sort_index()
axes[0].bar(year_counts.index.astype(int), year_counts.values, color='#457B9D', edgecolor='white')
axes[0].set_title('Reviews by Year of Last Review')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Listings')
for x, y in zip(year_counts.index.astype(int), year_counts.values):
    axes[0].text(x, y + 30, f'{y:,}', ha='center', fontsize=8)

month_counts = reviewed['review_month'].value_counts().sort_index()
month_names  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(month_names, month_counts.values, color='#2A9D8F', edgecolor='white')
axes[1].set_title('Review Activity by Month (Seasonality)')
axes[1].set_xlabel('Month of Last Review')
axes[1].set_ylabel('Number of Listings')

plt.tight_layout()
plt.show()


**Insights:**
1. **2019 dominates with 25,202 reviews** — confirming the dataset is a 2019 snapshot with recency bias toward most recently active listings.
2. **June is the peak review month** with 13,584 listings last reviewed in June — confirming summer as peak NYC tourism season.
3. **February is the trough** (770 listings) — consistent with NYC's off-peak winter tourism.
4. Year-over-year growth is exponential: 2015→2016 +94%, 2017→2018 +89% — classic platform growth curve.

---
## 3.9 Revenue Proxy & Occupancy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

active_rev = df[df['revenue_proxy'] > 0]
sns.histplot(np.log1p(active_rev['revenue_proxy']), bins=60,
             color='#8338EC', kde=True, ax=axes[0])
axes[0].set_title('log1p(Revenue Proxy) Distribution')
axes[0].set_xlabel('log1p(Price × Availability)')

sample = df.sample(4000, random_state=42)
scatter = axes[1].scatter(
    sample['price'], sample['occupancy_rate_est'] * 100,
    c=sample['neighbourhood_group'].map(BOROUGH_PALETTE),
    alpha=0.4, s=15, edgecolors='none'
)
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Occupancy Rate (%)')
axes[1].set_title('Price vs Occupancy (4k samples)')
axes[1].set_xlim(0, 500)
legend_elements = [Patch(facecolor=BOROUGH_PALETTE[b], label=b) for b in borough_order]
axes[1].legend(handles=legend_elements, title='Borough', fontsize=8)

plt.tight_layout()
plt.show()


**Insights:**
1. **Revenue proxy (price × availability) is also right-skewed** — a small segment of listings (~5%) generates disproportionately high potential revenue.
2. **Manhattan listings cluster at high price + moderate occupancy**, while Brooklyn occupies high occupancy + moderate price.
3. **Weak negative price-occupancy correlation** (r ≈ −0.12) confirms that higher-priced listings are not booked proportionally less — NYC market is price-inelastic at scale.

---
## 3.10 Correlation Heatmap

In [ ]:
corr_cols = [
    'price', 'log_price', 'minimum_nights', 'review_count',
    'review_rate_month', 'availability_365', 'revenue_proxy',
    'occupancy_rate_est', 'host_listing_count', 'is_multi_lister', 'has_reviews'
]
corr_matrix = df[corr_cols].corr(method='pearson')

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Pearson Correlation Matrix — Cleaned Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

corr_pairs = (
    corr_matrix.where(~mask)
    .stack().reset_index()
    .rename(columns={'level_0':'var1','level_1':'var2',0:'r'})
)
corr_pairs['abs_r'] = corr_pairs['r'].abs()
corr_pairs = corr_pairs[corr_pairs['var1'] != corr_pairs['var2']]


**Key correlations:**
- `availability_365` ↔ `occupancy_rate_est`: **r = −1.00** — by construction (occupancy = 1 − avail/365)
- `price` ↔ `revenue_proxy`: **r = +0.64** — price is the primary revenue driver, availability is secondary
- `review_count` ↔ `review_rate_month`: **r = +0.59** — active listings accumulate reviews at consistent monthly rates
- `log_price` ↔ `price`: **r = +0.89** — confirms log transform captures near-all price variance
- `host_listing_count` ↔ `availability_365`: **r = +0.23** — professional hosts keep listings more available (commercial strategy)

---
## 3.11 Outlier Deep-Dive

In [ ]:
price_arr = df['price'].to_numpy()
nights_arr = df['minimum_nights'].to_numpy()
rev_arr = df['revenue_proxy'].to_numpy()

def iqr_bounds(arr):
    q1, q3 = np.percentile(arr, [25, 75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col, arr in [('price', price_arr), ('minimum_nights', nights_arr), ('revenue_proxy', rev_arr)]:
    lo, hi = iqr_bounds(arr)
    n_out = ((arr < lo) | (arr > hi)).sum()


In [ ]:
luxury_df = df[df['is_luxury'] == 1]


**Insights:**
1. **Price outliers (IQR method): ~12%** of listings — confirms right tail is significant even post-cap.
2. **Luxury listings (>p99, price capped) are 97% Entire home/apt** — extremely rare for private or shared rooms to reach the luxury tier.
3. **Manhattan accounts for ~75% of luxury listings** — premium pricing is almost exclusively a Manhattan phenomenon.

---
## 3.12 EDA Summary & Hypotheses for Notebook 04

### Key Findings

| # | Finding | Business Implication |
|---|---|---|
| 1 | Price is right-skewed; median ($106) ≪ mean ($153) | Use median for dashboard KPI, mean for revenue modelling |
| 2 | Brooklyn has highest occupancy (72.5%) at lower price | Best price-to-occupancy ratio — high-ROI market for hosts |
| 3 | 35.9% of listings have 0 availability | Dashboard must filter for 'active' listings (avail > 0) |
| 4 | June is peak season; February is trough | Seasonal pricing strategy — ~18× more listings reviewed in June vs Feb |
| 5 | 33.9% multi-listing hosts; one host has 327 listings | Commercialisation is material — warrants regulatory concern |
| 6 | Revenue proxy driven by price (r=0.64) not occupancy | Price optimisation > availability optimisation for revenue |
| 7 | Manhattan accounts for ~75% of luxury listings | Luxury tier is a Manhattan-only dashboard filter |
| 8 | review_year skewed toward 2019 (25,202 of 38,833) | Dataset is effectively a 2019 snapshot |

### Hypotheses to Test in `04_statistical_analysis.ipynb`

| Hypothesis | Test |
|---|---|
| H1: Manhattan prices are significantly higher than Brooklyn | Welch's t-test / Mann-Whitney U |
| H2: Entire home/apt prices differ significantly from private rooms | Welch's t-test |
| H3: Multi-listers earn significantly higher revenue proxy | Mann-Whitney U |
| H4: Price is positively associated with borough prestige (ordinal) | Spearman correlation / Kruskal-Wallis |
| H5: log_price can be predicted from room_type + borough + availability | OLS regression |

> **Proceed to** `04_statistical_analysis.ipynb`